In [ ]:
!pip -q install -U gdown


In [ ]:
import requests

BASE = "http://13.220.223.194:8000"

r = requests.get(f"{BASE}/health", timeout=10)
print(r.status_code, r.json())


200 {'status': 'ok', 'gpu': True}


In [ ]:
import requests

BASE = "http://13.220.223.194:8000"
file_path = "/content/Bandido - Vamos Amigos _Eurodance Version_.wav"

with open(file_path, "rb") as f:
    files = {"file": ("Bandido.wav", f, "audio/wav")}
    r = requests.post(f"{BASE}/analyze", files=files, timeout=300)

print("status:", r.status_code)
print(r.json())


status: 200
{'filename': 'Bandido.wav', 'processing_sec': 1.73, 'segments': [{'start': 0.0, 'end': 7.2, 'time': '00:00.000 → 00:07.200', 'duration': 7.2, 'label': 'verse'}, {'start': 7.2, 'end': 20.88, 'time': '00:07.200 → 00:20.881', 'duration': 13.68, 'label': 'chorus'}, {'start': 20.88, 'end': 35.28, 'time': '00:20.881 → 00:35.281', 'duration': 14.4, 'label': 'chorus'}], 'structure': 'verse-chorus-chorus', 'logits': {'frame_rate': 8.333, 'T': 294, 'C': 128, 'time_sec': [0.0, 0.12000480019200768, 0.24000960038401536, 0.360014400576023, 0.48001920076803073, 0.6000240009600384, 0.720028801152046, 0.8400336013440537, 0.9600384015360615, 1.080043201728069, 1.2000480019200768, 1.3200528021120845, 1.440057602304092, 1.5600624024960998, 1.6800672026881074, 1.8000720028801152, 1.920076803072123, 2.0400816032641305, 2.160086403456138, 2.280091203648146, 2.4000960038401535, 2.520100804032161, 2.640105604224169, 2.7601104044161766, 2.880115204608184, 3.000120004800192, 3.1201248049921997, 3.240

In [ ]:
!pip install faiss-cpu tqdm librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.7 MB/s eta 0:00:00


In [ ]:
import json
data = r.json()

out_path = "result.json"
with open(out_path, "w", encoding="utf-8") as fp:
    json.dump(data, fp, ensure_ascii=False, indent=2)

In [ ]:
FILE_ID = "https://drive.google.com/drive/folders/1bgvD0m3sdFKORlwoelBoRqVacLW4HskF"
!gdown --id $FILE_ID -O dataset.zip


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=https://drive.google.com/drive/folders/1bgvD0m3sdFKORlwoelBoRqVacLW4HskF

but Gdown can't. Please check connections and permissions.


In [ ]:
!unzip -q "original.zip" -d "/content"
!ls -la /content/original | head


[original.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of original.zip or
        original.zip.zip, and cannot find original.zip.ZIP, period.
ls: cannot access '/content/original': No such file or directory


In [ ]:
!unzip -q "/content/original.zip" -d "/content/original"


[/content/original.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/original.zip or
        /content/original.zip.zip, and cannot find /content/original.zip.ZIP, period.


In [ ]:
!ls -lah /content/original.zip
!file /content/original.zip
!head -c 200 /content/original.zip | cat


-rw-r--r-- 1 root root 69M Jan 29 11:19 /content/original.zip
/content/original.zip: Zip archive data, at least v2.0 to extract, compression method=store
 ��yi��yi��yiux �     ��w|W

In [ ]:
!unzip -q "comparison.zip" -d "/content/comparison"
!ls -la /content/comparison | head

[comparison.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of comparison.zip or
        comparison.zip.zip, and cannot find comparison.zip.ZIP, period.
ls: cannot access '/content/comparison': No such file or directory


In [ ]:
"""
Гібридний пошук схожих пісень через API аналізу структури.

API повертає:
- segments: частини пісні (verse, chorus, bridge...)
- logits: матриця (T, 128) — фічі по часу
- structure: текстова структура ("verse-chorus-chorus")

Пошук:
1. FAISS — швидкий відбір по усередненому вектору logits
2. DTW — точне порівняння послідовностей logits

Запуск:
    pip install requests numpy pandas faiss-cpu scipy tqdm
    python api_song_search.py
"""

import json
import time
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import numpy as np
import pandas as pd
import faiss
from scipy.spatial.distance import cdist
from tqdm import tqdm


# ============================================================
# НАЛАШТУВАННЯ
# ============================================================

@dataclass
class Config:
    # API
    api_url: str = "http://13.220.223.194:8000/analyze"
    api_timeout: int = 300
    max_workers: int = 4  # паралельні запити до API

    # Шляхи
    origin_dir: Path = Path("/content/original")
    comparison_dir: Path = Path("/content/comparison")
    pairs_csv: Path = Path("/content/song_pairs.csv")
    cache_dir: Path = Path("./cache_api")

    # Гібридний пошук
    faiss_candidates: int = 50

    # Кеш
    use_cache: bool = True


# ============================================================
# API КЛІЄНТ
# ============================================================

def analyze_file(filepath: Path, api_url: str, timeout: int) -> dict | None:
    """
    Відправляє файл на API і повертає результат.

    Повертає dict з ключами:
    - logits_matrix: np.ndarray (T, 128)
    - segments: list of dicts
    - structure: str
    """
    try:
        with open(filepath, "rb") as f:
            files = {"file": (filepath.name, f, "audio/wav")}
            response = requests.post(api_url, files=files, timeout=timeout)

        if response.status_code != 200:
            print(f"[API ERROR] {filepath.name}: status {response.status_code}")
            return None

        data = response.json()

        # Конвертуємо logits в матрицю (T, C)
        logits = data.get("logits", {})
        T = logits.get("T", 0)
        C = logits.get("C", 0)
        values = logits.get("values", [])

        if T == 0 or C == 0 or not values:
            print(f"[API ERROR] {filepath.name}: empty logits")
            return None

        # values — це плоский список, reshape в (T, C)
        matrix = np.array(values, dtype=np.float32).reshape(T, C)

        return {
            "logits_matrix": matrix,
            "segments": data.get("segments", []),
            "structure": data.get("structure", ""),
            "frame_rate": logits.get("frame_rate", 8.333),
        }

    except Exception as e:
        print(f"[API ERROR] {filepath.name}: {e}")
        return None


# ============================================================
# DTW (Dynamic Time Warping)
# ============================================================

def dtw_distance(seq1: np.ndarray, seq2: np.ndarray) -> float:
    """
    DTW відстань між послідовностями.

    seq1: (T1, D)
    seq2: (T2, D)
    """
    n, m = len(seq1), len(seq2)

    # Косинусна відстань між кожною парою фреймів
    cost = cdist(seq1, seq2, metric="cosine")

    # DP таблиця
    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i, j] = cost[i-1, j-1] + min(
                dp[i-1, j],
                dp[i, j-1],
                dp[i-1, j-1]
            )

    return dp[n, m] / (n + m)


# ============================================================
# ГІБРИДНИЙ ІНДЕКС
# ============================================================

class HybridIndex:
    """FAISS для швидкого відбору + DTW для точного порівняння."""

    def __init__(self, n_candidates: int = 50):
        self.n_candidates = n_candidates
        self.faiss_index = None
        self.sequences = []
        self.metadata = []

    def build(self, means: np.ndarray, sequences: list, metadata: list):
        """Будує індекс."""
        # Нормалізація для cosine similarity
        norms = np.linalg.norm(means, axis=1, keepdims=True)
        means_norm = means / (norms + 1e-12)

        self.faiss_index = faiss.IndexFlatIP(means_norm.shape[1])
        self.faiss_index.add(means_norm.astype("float32"))

        self.sequences = sequences
        self.metadata = metadata

        print(f"Індекс побудовано: {len(metadata)} треків")

    def search(self, query_mean: np.ndarray, query_seq: np.ndarray) -> tuple:
        """
        Пошук найближчого треку.

        Повертає: (metadata_dict, similarity_score)
        """
        # Нормалізуємо запит
        query_mean = query_mean / (np.linalg.norm(query_mean) + 1e-12)
        query_mean = query_mean.reshape(1, -1).astype("float32")

        # FAISS: швидкий відбір кандидатів
        n_search = min(self.n_candidates, len(self.sequences))
        _, indices = self.faiss_index.search(query_mean, n_search)

        # DTW: точне порівняння
        best_idx = None
        best_dist = float("inf")

        for idx in indices[0]:
            dist = dtw_distance(query_seq, self.sequences[idx])
            if dist < best_dist:
                best_dist = dist
                best_idx = idx

        similarity = 1.0 / (1.0 + best_dist)
        return self.metadata[best_idx], similarity


# ============================================================
# КЕШУВАННЯ
# ============================================================

def get_cache_path(cache_dir: Path, filename: str) -> Path:
    """Шлях до кешу для файлу."""
    return cache_dir / f"{filename}.json"


def save_to_cache(cache_dir: Path, filename: str, data: dict):
    """Зберігає результат API в кеш."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = get_cache_path(cache_dir, filename)

    # Конвертуємо numpy в list для JSON
    cache_data = {
        "logits_matrix": data["logits_matrix"].tolist(),
        "segments": data["segments"],
        "structure": data["structure"],
        "frame_rate": data["frame_rate"],
    }
    cache_path.write_text(json.dumps(cache_data))


def load_from_cache(cache_dir: Path, filename: str) -> dict | None:
    """Завантажує з кешу."""
    cache_path = get_cache_path(cache_dir, filename)

    if not cache_path.exists():
        return None

    try:
        cache_data = json.loads(cache_path.read_text())
        cache_data["logits_matrix"] = np.array(cache_data["logits_matrix"], dtype=np.float32)
        return cache_data
    except:
        return None


# ============================================================
# ОБРОБКА ПАПКИ
# ============================================================

def process_folder(folder: Path, cfg: Config) -> tuple:
    """
    Обробляє всі .wav файли в папці через API.

    Повертає: (means, sequences, metadata)
    """
    files = sorted(folder.rglob("*.wav"))
    print(f"Знайдено {len(files)} файлів в {folder}")

    means = []
    sequences = []
    metadata = []

    # Функція для обробки одного файлу
    def process_one(filepath: Path) -> tuple:
        filename = filepath.name

        # Пробуємо кеш
        if cfg.use_cache:
            cached = load_from_cache(cfg.cache_dir, filename)
            if cached:
                return filepath, cached

        # Запит до API
        result = analyze_file(filepath, cfg.api_url, cfg.api_timeout)

        # Зберігаємо в кеш
        if result and cfg.use_cache:
            save_to_cache(cfg.cache_dir, filename, result)

        return filepath, result

    # Паралельна обробка
    with ThreadPoolExecutor(max_workers=cfg.max_workers) as executor:
        futures = {executor.submit(process_one, f): f for f in files}

        for future in tqdm(as_completed(futures), total=len(files), desc="Обробка"):
            filepath, result = future.result()

            if result is None:
                continue

            matrix = result["logits_matrix"]

            # Mean ембединг для FAISS
            mean = matrix.mean(axis=0)

            means.append(mean)
            sequences.append(matrix)
            metadata.append({
                "name": filepath.name,
                "path": str(filepath),
                "structure": result["structure"],
            })

    if not means:
        raise RuntimeError("Не вдалось обробити жодного файлу")

    means = np.stack(means)
    return means, sequences, metadata


# ============================================================
# ГОЛОВНА ФУНКЦІЯ
# ============================================================

def find_wav_files(folder: Path) -> dict:
    """Мапа: ім'я файлу -> шлях."""
    return {p.name: p for p in folder.rglob("*.wav") if p.is_file()}


def ensure_wav(name: str) -> str:
    """Додає .wav якщо треба."""
    return name if name.lower().endswith(".wav") else name + ".wav"


def run(cfg: Config):
    """Запускає експеримент."""

    print("=" * 60)
    print("ГІБРИДНИЙ ПОШУК ЧЕРЕЗ API")
    print(f"API: {cfg.api_url}")
    print(f"Кандидатів FAISS: {cfg.faiss_candidates}")
    print("=" * 60)

    # 1. Обробляємо comparison папку
    print("\n[1] Обробка comparison...")
    means, sequences, metadata = process_folder(cfg.comparison_dir, cfg)

    # 2. Будуємо індекс
    print("\n[2] Будую індекс...")
    index = HybridIndex(n_candidates=cfg.faiss_candidates)
    index.build(means, sequences, metadata)

    # 3. Мапа файлів
    origin_map = find_wav_files(cfg.origin_dir)
    comp_map = {m["name"]: m for m in metadata}

    # 4. Читаємо пари з CSV
    print("\n[3] Читаю пари...")
    df = pd.read_csv(cfg.pairs_csv)

    # 5. Перевіряємо пари
    print("\n[4] Пошук...")
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        ori_name = ensure_wav(row["ori_title"])
        comp_name = ensure_wav(row["comp_title"])

        ori_path = origin_map.get(ori_name)

        if ori_path is None:
            results.append({
                "ori": ori_name,
                "expected": comp_name,
                "predicted": None,
                "score": None,
                "match": False,
                "status": "ORI_NOT_FOUND"
            })
            continue

        # Отримуємо ембединг запиту
        if cfg.use_cache:
            cached = load_from_cache(cfg.cache_dir, ori_name)
            if cached:
                query_matrix = cached["logits_matrix"]
            else:
                result = analyze_file(ori_path, cfg.api_url, cfg.api_timeout)
                if result:
                    save_to_cache(cfg.cache_dir, ori_name, result)
                    query_matrix = result["logits_matrix"]
                else:
                    results.append({
                        "ori": ori_name,
                        "expected": comp_name,
                        "predicted": None,
                        "score": None,
                        "match": False,
                        "status": "API_ERROR"
                    })
                    continue
        else:
            result = analyze_file(ori_path, cfg.api_url, cfg.api_timeout)
            if not result:
                results.append({
                    "ori": ori_name,
                    "expected": comp_name,
                    "predicted": None,
                    "score": None,
                    "match": False,
                    "status": "API_ERROR"
                })
                continue
            query_matrix = result["logits_matrix"]

        query_mean = query_matrix.mean(axis=0)

        # Пошук
        match, score = index.search(query_mean, query_matrix)

        results.append({
            "ori": ori_name,
            "expected": comp_name,
            "predicted": match["name"],
            "score": round(score, 4),
            "match": match["name"] == comp_name,
            "status": "OK"
        })

    # 6. Результати
    results_df = pd.DataFrame(results)

    valid = results_df[results_df["status"] == "OK"]
    accuracy = valid["match"].mean() if len(valid) > 0 else 0

    print("\n" + "=" * 60)
    print("РЕЗУЛЬТАТИ")
    print("=" * 60)
    print(f"Всього пар: {len(results_df)}")
    print(f"Перевірено: {len(valid)}")
    print(f"Точність: {accuracy:.2%}")

    # Статистика по статусах
    print(f"\nСтатуси:")
    print(results_df["status"].value_counts().to_string())

    # Помилки
    errors = valid[~valid["match"]].sort_values("score", ascending=False)
    if len(errors) > 0:
        print(f"\nПомилки ({len(errors)}):")
        print(errors[["ori", "expected", "predicted", "score"]].head(10).to_string(index=False))

    # Зберігаємо
    output = Path("./api_search_results.csv")
    results_df.to_csv(output, index=False)
    print(f"\nЗбережено: {output}")

    return results_df

cfg = Config()

    # Налаштування:
cfg.faiss_candidates = 3  # більше кандидатів = точніше, але повільніше
cfg.max_workers = 1      # більше паралельних запитів
cfg.use_cache = False       # вимкнути кеш
run(cfg)

ГІБРИДНИЙ ПОШУК ЧЕРЕЗ API
API: http://13.220.223.194:8000/analyze
Кандидатів FAISS: 3

[1] Обробка comparison...
Знайдено 7 файлів в /content/comparison


Обробка:  14%|█▍        | 1/7 [00:01<00:08,  1.36s/it]

[API ERROR] Bandido - Vamos Amigos _Eurodance Version_.wav: empty logits


Обробка:  29%|██▊       | 2/7 [00:03<00:07,  1.54s/it]

[API ERROR] Ed Sheeran - Bad Habits _Official Lyric Video_.wav: empty logits


...

Обробка:  86%|████████▌ | 6/7 [00:07<00:01,  1.22s/it]

[API ERROR] _DANCE_ 싸이 _PSY_ - 챔피언.wav: empty logits


Обробка: 100%|██████████| 7/7 [00:09<00:00,  1.35s/it]

[API ERROR] ТНМК _ _Люба_ Люба_.wav: empty logits


RuntimeError: Не вдалось обробити жодного файлу

In [ ]:
"""
Гібридний пошук схожих пісень через API аналізу структури.

API повертає:
- segments: частини пісні (verse, chorus, bridge...)
- boundary_logits: (T,) — ймовірності меж сегментів
- function_logits_topk: топ-8 функцій з їх значеннями (T,) кожна

Запуск:
    pip install requests numpy pandas faiss-cpu scipy tqdm
    python api_song_search_v3.py
"""

import json
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import numpy as np
import pandas as pd
import faiss
from scipy.spatial.distance import cdist
from tqdm import tqdm


# ============================================================
# НАЛАШТУВАННЯ
# ============================================================

@dataclass
class Config:
    # API
    api_url: str = "http://13.220.223.194:8000/analyze"
    api_timeout: int = 300
    max_workers: int = 4

    # Шляхи
    origin_dir: Path = Path("/content/original")
    comparison_dir: Path = Path("/content/comparison")
    pairs_csv: Path = Path("/content/song_pairs.csv")
    cache_dir: Path = Path("./cache_api")

    # Пошук
    faiss_candidates: int = 5
    use_cache: bool = True


# ============================================================
# ПАРСИНГ API ВІДПОВІДІ
# ============================================================

def parse_api_response(data: dict) -> dict | None:
    """
    Парсить відповідь API і створює матрицю фіч.

    Збираємо:
    - boundary_logits: (T,) — межі сегментів
    - function_logits_topk: (T, K) — значення для кожної функції

    Результат: матриця (T, K+1)
    """
    logits = data.get("logits", {})

    # Boundary logits
    boundary = logits.get("boundary_logits", [])
    if not boundary:
        return None

    T = len(boundary)
    boundary = np.array(boundary, dtype=np.float32).reshape(T, 1)

    # Function logits (топ-K функцій)
    functions = logits.get("function_logits_topk", [])
    if not functions:
        return None

    # Збираємо значення кожної функції в колонки
    func_matrix = []
    func_names = []

    for func in functions:
        values = func.get("values", [])
        name = func.get("name", "unknown")

        if len(values) == T:
            func_matrix.append(values)
            func_names.append(name)

    if not func_matrix:
        return None

    # (T, K) — кожна колонка це одна функція
    func_matrix = np.array(func_matrix, dtype=np.float32).T

    # Об'єднуємо: (T, K+1) = boundary + functions
    matrix = np.hstack([boundary, func_matrix])

    return {
        "matrix": matrix,  # (T, K+1)
        "segments": data.get("segments", []),
        "structure": data.get("structure", ""),
        "function_names": func_names,
        "T": T,
    }


# ============================================================
# API КЛІЄНТ
# ============================================================

def analyze_file(filepath: Path, api_url: str, timeout: int) -> dict | None:
    """Відправляє файл на API."""
    try:
        with open(filepath, "rb") as f:
            files = {"file": (filepath.name, f, "audio/wav")}
            response = requests.post(api_url, files=files, timeout=timeout)

        if response.status_code != 200:
            print(f"[HTTP {response.status_code}] {filepath.name}")
            return None

        data = response.json()
        result = parse_api_response(data)

        if result is None:
            print(f"[PARSE ERROR] {filepath.name}")
            return None

        return result

    except Exception as e:
        print(f"[ERROR] {filepath.name}: {e}")
        return None


# ============================================================
# DTW
# ============================================================

def dtw_distance(seq1: np.ndarray, seq2: np.ndarray) -> float:
    """DTW відстань між послідовностями."""
    n, m = len(seq1), len(seq2)
    cost = cdist(seq1, seq2, metric="cosine")

    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i, j] = cost[i-1, j-1] + min(dp[i-1, j], dp[i, j-1], dp[i-1, j-1])

    return dp[n, m] / (n + m)


# ============================================================
# ГІБРИДНИЙ ІНДЕКС
# ============================================================

class HybridIndex:
    def __init__(self, n_candidates: int = 50):
        self.n_candidates = n_candidates
        self.faiss_index = None
        self.sequences = []
        self.metadata = []

    def build(self, means: np.ndarray, sequences: list, metadata: list):
        norms = np.linalg.norm(means, axis=1, keepdims=True)
        means_norm = means / (norms + 1e-12)

        self.faiss_index = faiss.IndexFlatIP(means_norm.shape[1])
        self.faiss_index.add(means_norm.astype("float32"))

        self.sequences = sequences
        self.metadata = metadata
        print(f"Індекс: {len(metadata)} треків, {means.shape[1]} фіч")

    def search(self, query_mean: np.ndarray, query_seq: np.ndarray) -> tuple:
        query_mean = query_mean / (np.linalg.norm(query_mean) + 1e-12)
        query_mean = query_mean.reshape(1, -1).astype("float32")

        n_search = min(self.n_candidates, len(self.sequences))
        _, indices = self.faiss_index.search(query_mean, n_search)

        best_idx, best_dist = None, float("inf")
        for idx in indices[0]:
            dist = dtw_distance(query_seq, self.sequences[idx])
            if dist < best_dist:
                best_dist = dist
                best_idx = idx

        return self.metadata[best_idx], 1.0 / (1.0 + best_dist)


# ============================================================
# КЕШ
# ============================================================

def cache_path(cache_dir: Path, filename: str) -> Path:
    return cache_dir / f"{filename}.npz"


def save_cache(cache_dir: Path, filename: str, data: dict):
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_path(cache_dir, filename)
    np.savez_compressed(
        path,
        matrix=data["matrix"],
        structure=data["structure"],
        segments=json.dumps(data["segments"]),
    )


def load_cache(cache_dir: Path, filename: str) -> dict | None:
    path = cache_path(cache_dir, filename)
    if not path.exists():
        return None
    try:
        loaded = np.load(path, allow_pickle=True)
        return {
            "matrix": loaded["matrix"],
            "structure": str(loaded["structure"]),
            "segments": json.loads(str(loaded["segments"])),
        }
    except:
        return None


# ============================================================
# ДОПОМІЖНІ
# ============================================================

def find_wav_files(folder: Path) -> list[Path]:
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob("*.wav") if p.is_file())


def ensure_wav(name: str) -> str:
    name = str(name)
    return name if name.lower().endswith(".wav") else name + ".wav"


def validate_paths(cfg: Config) -> bool:
    print("\n" + "=" * 50)
    print("ПЕРЕВІРКА ШЛЯХІВ")
    print("=" * 50)

    ok = True

    ori_files = find_wav_files(cfg.origin_dir)
    comp_files = find_wav_files(cfg.comparison_dir)

    print(f"Origin: {cfg.origin_dir}")
    print(f"  → {len(ori_files)} .wav файлів")
    ok = ok and len(ori_files) > 0

    print(f"Comparison: {cfg.comparison_dir}")
    print(f"  → {len(comp_files)} .wav файлів")
    ok = ok and len(comp_files) > 0

    print(f"CSV: {cfg.pairs_csv}")
    if cfg.pairs_csv.exists():
        df = pd.read_csv(cfg.pairs_csv)
        print(f"  → {len(df)} пар")
    else:
        print(f"  → НЕ ІСНУЄ!")
        ok = False

    return ok


# ============================================================
# ОБРОБКА
# ============================================================

def process_folder(folder: Path, cfg: Config) -> tuple:
    """Обробляє папку через API."""
    files = find_wav_files(folder)
    print(f"Обробка {len(files)} файлів...")

    if not files:
        raise RuntimeError(f"Не знайдено .wav у {folder}")

    means, sequences, metadata = [], [], []

    def process_one(filepath: Path):
        if cfg.use_cache:
            cached = load_cache(cfg.cache_dir, filepath.name)
            if cached:
                return filepath, cached

        result = analyze_file(filepath, cfg.api_url, cfg.api_timeout)

        if result and cfg.use_cache:
            save_cache(cfg.cache_dir, filepath.name, result)

        return filepath, result

    with ThreadPoolExecutor(max_workers=cfg.max_workers) as executor:
        futures = {executor.submit(process_one, f): f for f in files}

        for future in tqdm(as_completed(futures), total=len(files), desc="API"):
            filepath, result = future.result()

            if result is None:
                continue

            matrix = result["matrix"]
            means.append(matrix.mean(axis=0))
            sequences.append(matrix)
            metadata.append({
                "name": filepath.name,
                "path": str(filepath),
                "structure": result["structure"],
            })

    if not means:
        raise RuntimeError("Не вдалось обробити жодного файлу")

    print(f"Успішно: {len(means)} файлів")
    return np.stack(means), sequences, metadata


def get_query_embedding(filepath: Path, cfg: Config) -> tuple | None:
    if cfg.use_cache:
        cached = load_cache(cfg.cache_dir, filepath.name)
        if cached:
            m = cached["matrix"]
            return m.mean(axis=0), m

    result = analyze_file(filepath, cfg.api_url, cfg.api_timeout)
    if result is None:
        return None

    if cfg.use_cache:
        save_cache(cfg.cache_dir, filepath.name, result)

    return result["matrix"].mean(axis=0), result["matrix"]


# ============================================================
# ГОЛОВНА
# ============================================================

def run(cfg: Config):
    if not validate_paths(cfg):
        print("\n❌ Виправ шляхи!")
        return None

    print("\n" + "=" * 50)
    print("ГІБРИДНИЙ ПОШУК")
    print("=" * 50)

    # 1. Comparison
    print("\n[1/4] Обробка comparison...")
    means, sequences, metadata = process_folder(cfg.comparison_dir, cfg)

    # 2. Індекс
    print("\n[2/4] Будую індекс...")
    index = HybridIndex(n_candidates=cfg.faiss_candidates)
    index.build(means, sequences, metadata)

    # 3. CSV
    print("\n[3/4] Читаю пари...")
    df = pd.read_csv(cfg.pairs_csv)
    origin_map = {p.name: p for p in find_wav_files(cfg.origin_dir)}

    # 4. Пошук
    print("\n[4/4] Пошук...")
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Пошук"):
        ori_name = ensure_wav(row["ori_title"])
        comp_name = ensure_wav(row["comp_title"])

        ori_path = origin_map.get(ori_name)

        if ori_path is None:
            results.append({
                "ori": ori_name,
                "expected": comp_name,
                "predicted": None,
                "score": None,
                "match": False,
                "status": "ORI_NOT_FOUND"
            })
            continue

        emb = get_query_embedding(ori_path, cfg)

        if emb is None:
            results.append({
                "ori": ori_name,
                "expected": comp_name,
                "predicted": None,
                "score": None,
                "match": False,
                "status": "API_ERROR"
            })
            continue

        query_mean, query_seq = emb
        match, score = index.search(query_mean, query_seq)

        results.append({
            "ori": ori_name,
            "expected": comp_name,
            "predicted": match["name"],
            "score": round(score, 4),
            "match": match["name"] == comp_name,
            "status": "OK"
        })

    # Результати
    results_df = pd.DataFrame(results)
    valid = results_df[results_df["status"] == "OK"]
    accuracy = valid["match"].mean() if len(valid) > 0 else 0

    print("\n" + "=" * 50)
    print("РЕЗУЛЬТАТИ")
    print("=" * 50)
    print(f"Всього: {len(results_df)}")
    print(f"Перевірено: {len(valid)}")
    print(f"Точність: {accuracy:.2%}")

    print(f"\nСтатуси:")
    print(results_df["status"].value_counts().to_string())

    errors = valid[~valid["match"]].sort_values("score", ascending=False)
    if len(errors) > 0:
        print(f"\nПомилки ({len(errors)}):")
        print(errors[["ori", "expected", "predicted", "score"]].head(10).to_string(index=False))

    output = Path("./results.csv")
    results_df.to_csv(output, index=False)
    print(f"\nЗбережено: {output}")

    return results_df


cfg = Config()

cfg.origin_dir = Path("/content/original")
cfg.comparison_dir = Path("/content/comparison")

run(cfg)


ПЕРЕВІРКА ШЛЯХІВ
Origin: /content/original
  → 19 .wav файлів
Comparison: /content/comparison
  → 16 .wav файлів
CSV: /content/song_pairs.csv
  → 16 пар

ГІБРИДНИЙ ПОШУК

[1/4] Обробка comparison...
Обробка 16 файлів...


API: 100%|██████████| 16/16 [00:00<00:00, 5041.61it/s]


Успішно: 16 файлів

[2/4] Будую індекс...
Індекс: 16 треків, 9 фіч

[3/4] Читаю пари...

[4/4] Пошук...


Пошук: 100%|██████████| 16/16 [00:19<00:00,  1.23s/it]


РЕЗУЛЬТАТИ
Всього: 16
Перевірено: 15
Точність: 13.33%

Статуси:
status
OK               15
ORI_NOT_FOUND     1

Помилки (13):
                                                                             ori                                                                         expected                                                           predicted  score
                                     Queen - Under Pressure _Official Video_.wav                            Vanilla Ice - Ice Ice Baby _Official Music Video_.wav                            Smoke On The Water _2024 Remastered_.wav 0.9964
                                        The Gap Band - Oops Upside Your Head.wav                    Mark Ronson - Uptown Funk _Official Video_ ft_ Bruno Mars.wav                            Smoke On The Water _2024 Remastered_.wav 0.9939
                    Jennifer Rush - The Power Of Love _Official Video_ _VOD_.wav               Céline Dion - The Power Of Love _Official Remastered HD Video_.wav 

,ori,expected,predicted,score,match,status
0,The Gap Band - Oops Upside Your Head.wav,Mark Ronson - Uptown Funk _Official Video_ ft_...,Smoke On The Water _2024 Remastered_.wav,0.9939,False,OK
1,Jennifer Rush - The Power Of Love _Official Vi...,Céline Dion - The Power Of Love _Official Rema...,Céline Dion - The Power Of Love _Official Rem...,0.9906,False,OK
2,Without you - Badfinger.wav,Mariah Carey - Without You _Official Lyric Vid...,Mariah Carey - Without You _Official Lyric Vid...,0.9920,True,OK
3,_Official Audio_ 이정현_Lee Jung-hyun_ - 와.wav,Bandido - Vamos Amigos _Eurodance Version_.wav,None,NaN,False,ORI_NOT_FOUND
4,Queen - Under Pressure _Official Video_.wav,Vanilla Ice - Ice Ice Baby _Official Music Vid...,Smoke On The Water _2024 Remastered_.wav,0.9964,False,OK
5,Joyful Noise - Official Music Video_ FLAME fea...,Katy Perry - Dark Horse _Lyrics_ ft_ Juicy J.wav,Vanilla Ice - Ice Ice Baby _Official Music Vid...,0.9813,False,OK
6,Astrud Gilberto - Maria Quiet.wav,Smoke On The Water _2024 Remastered_.wav,_DANCE_ 싸이 _PSY_ - 챔피언.wav,0.9870,False,OK
7,Crazy Frog - Axel F _Official Video_.wav,_DANCE_ 싸이 _PSY_ - 챔피언.wav,Mark Ronson - Uptown Funk _Official Video_ ft_...,0.9794,False,OK
8,Sting - Shape of My Heart _Official Music Vide...,Juice WRLD - Lucid Dreams _Official Music Vide...,Juice WRLD - Lucid Dreams _Official Music Vide...,0.9962,True,OK
9,September - Cry For You.wav,Ed Sheeran - Bad Habits _Official Lyric Video_...,Bandido - Vamos Amigos _Eurodance Version_.wav,0.9666,False,OK


In [ ]:
"""
Пошук схожих пісень через API з покращеними ембедингами.

Ембединг включає:
- Позиційні фічі (структура по частинах)
- Статистики (std, max, тренди)
- Інформацію про переходи
- Домінантні функції

Запуск:
    pip install requests numpy pandas faiss-cpu scipy tqdm
    python rich_embedding_search.py
"""

import json
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import numpy as np
import pandas as pd
import faiss
from scipy.spatial.distance import cdist
from tqdm import tqdm


# ============================================================
# НАЛАШТУВАННЯ
# ============================================================

@dataclass
class Config:
    # API
    api_url: str = "http://13.220.223.194:8000/analyze"
    api_timeout: int = 300
    max_workers: int = 4

    # Шляхи
    origin_dir: Path = Path("/content/original")
    comparison_dir: Path = Path("/content/comparison")
    pairs_csv: Path = Path("/content/song_pairs.csv")
    cache_dir: Path = Path("./cache_api")

    # Ембединг
    n_parts: int = 4          # на скільки частин ділити трек
    use_dtw: bool = True      # чи використовувати DTW
    faiss_candidates: int = 30  # кандидатів для DTW (менше бо ембединг кращий)

    use_cache: bool = True


# ============================================================
# ПАРСИНГ API
# ============================================================

def parse_api_response(data: dict) -> dict | None:
    """Парсить відповідь API в матрицю (T, 9)."""
    logits = data.get("logits", {})

    boundary = logits.get("boundary_logits", [])
    if not boundary:
        return None

    T = len(boundary)
    boundary = np.array(boundary, dtype=np.float32).reshape(T, 1)

    functions = logits.get("function_logits_topk", [])
    if not functions:
        return None

    func_matrix = []
    func_names = []

    for func in functions:
        values = func.get("values", [])
        if len(values) == T:
            func_matrix.append(values)
            func_names.append(func.get("name", "unknown"))

    if not func_matrix:
        return None

    func_matrix = np.array(func_matrix, dtype=np.float32).T
    matrix = np.hstack([boundary, func_matrix])

    return {
        "matrix": matrix,
        "segments": data.get("segments", []),
        "structure": data.get("structure", ""),
        "func_names": ["boundary"] + func_names,
    }


# ============================================================
# БАГАТИЙ ЕМБЕДИНГ
# ============================================================

def compute_rich_embedding(matrix: np.ndarray, n_parts: int = 4) -> np.ndarray:
    """
    Створює багатий ембединг що зберігає структуру пісні.

    Включає:
    1. Позиційні mean (профіль кожної частини)
    2. Глобальні статистики (std, max)
    3. Позиції максимумів (де кульмінації)
    4. Тренди (як змінюється від початку до кінця)
    5. Активність переходів (скільки змін)
    6. Домінантні функції в кожній частині

    Args:
        matrix: (T, D) — сирі дані з API
        n_parts: на скільки частин ділити

    Returns:
        embedding: (N,) — багатий вектор фіч
    """
    T, D = matrix.shape
    part_size = T // n_parts
    features = []

    # ===== 1. ПОЗИЦІЙНІ MEAN =====
    # Профіль кожної частини треку
    # [intro | early | middle | end]
    for i in range(n_parts):
        start = i * part_size
        end = start + part_size if i < n_parts - 1 else T
        part_mean = matrix[start:end].mean(axis=0)
        features.append(part_mean)
    # Розмір: n_parts × D

    # ===== 2. ГЛОБАЛЬНІ СТАТИСТИКИ =====
    features.append(matrix.std(axis=0))   # варіативність
    features.append(matrix.max(axis=0))   # пікові значення
    features.append(matrix.min(axis=0))   # мінімальні значення
    # Розмір: 3 × D

    # ===== 3. ПОЗИЦІЇ МАКСИМУМІВ =====
    # Де знаходиться пік кожної фічі (0 = початок, 1 = кінець)
    max_positions = matrix.argmax(axis=0) / T
    features.append(max_positions)
    # Розмір: D

    # ===== 4. ТРЕНДИ =====
    # Як змінюються фічі від початку до кінця
    first_quarter = matrix[:T//4].mean(axis=0)
    last_quarter = matrix[-T//4:].mean(axis=0)
    trend = last_quarter - first_quarter
    features.append(trend)
    # Розмір: D

    # ===== 5. АКТИВНІСТЬ ПЕРЕХОДІВ =====
    # Скільки "руху" в кожній фічі
    diffs = np.abs(np.diff(matrix, axis=0))
    activity = diffs.mean(axis=0)  # середня зміна
    features.append(activity)

    # Де найбільше змін? (гістограма)
    change_magnitude = diffs.sum(axis=1)
    hist, _ = np.histogram(
        np.arange(len(change_magnitude)),
        bins=n_parts,
        weights=change_magnitude
    )
    hist = hist / (hist.sum() + 1e-12)
    features.append(hist)
    # Розмір: D + n_parts

    # ===== 6. BOUNDARY PEAKS =====
    # Топ-N найсильніших меж (де сегменти)
    boundary = matrix[:, 0]
    top_k = min(5, len(boundary))
    top_boundary_values = np.sort(boundary)[-top_k:]
    features.append(top_boundary_values)

    # Позиції цих меж
    top_boundary_positions = np.argsort(boundary)[-top_k:] / T
    top_boundary_positions = np.sort(top_boundary_positions)
    features.append(top_boundary_positions)
    # Розмір: 2 × top_k

    # ===== 7. ДОМІНАНТНА ФУНКЦІЯ В КОЖНІЙ ЧАСТИНІ =====
    # One-hot: яка функція найсильніша
    n_funcs = D - 1  # без boundary
    for i in range(n_parts):
        start = i * part_size
        end = start + part_size if i < n_parts - 1 else T
        func_means = matrix[start:end, 1:].mean(axis=0)
        dominant_idx = func_means.argmax()
        one_hot = np.zeros(n_funcs)
        one_hot[dominant_idx] = 1
        features.append(one_hot)
    # Розмір: n_parts × n_funcs

    # Збираємо все
    embedding = np.concatenate(features)

    # L2 нормалізація
    embedding = embedding / (np.linalg.norm(embedding) + 1e-12)

    return embedding.astype(np.float32)


def get_embedding_description(n_parts: int, D: int) -> str:
    """Повертає опис структури ембедингу."""
    n_funcs = D - 1
    top_k = 5

    sizes = [
        f"positional_means: {n_parts} × {D} = {n_parts * D}",
        f"std, max, min: 3 × {D} = {3 * D}",
        f"max_positions: {D}",
        f"trend: {D}",
        f"activity: {D}",
        f"change_hist: {n_parts}",
        f"boundary_peaks: {2 * top_k}",
        f"dominant_funcs: {n_parts} × {n_funcs} = {n_parts * n_funcs}",
    ]

    total = (n_parts * D) + (3 * D) + D + D + D + n_parts + (2 * top_k) + (n_parts * n_funcs)

    return "\n".join(sizes) + f"\n\nТОТАЛ: {total} фіч"


# ============================================================
# DTW (опціонально)
# ============================================================

def dtw_distance(seq1: np.ndarray, seq2: np.ndarray) -> float:
    """DTW відстань між послідовностями."""
    n, m = len(seq1), len(seq2)

    # Оптимізація: зменшуємо роздільність для швидкості
    step = max(1, min(n, m) // 100)
    seq1 = seq1[::step]
    seq2 = seq2[::step]
    n, m = len(seq1), len(seq2)

    cost = cdist(seq1, seq2, metric="cosine")

    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i, j] = cost[i-1, j-1] + min(dp[i-1, j], dp[i, j-1], dp[i-1, j-1])

    return dp[n, m] / (n + m)


# ============================================================
# FAISS ІНДЕКС
# ============================================================

class SearchIndex:
    def __init__(self, use_dtw: bool = True, n_candidates: int = 30):
        self.use_dtw = use_dtw
        self.n_candidates = n_candidates
        self.faiss_index = None
        self.sequences = []  # для DTW
        self.metadata = []

    def build(self, embeddings: np.ndarray, sequences: list, metadata: list):
        """Будує індекс."""
        self.faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
        self.faiss_index.add(embeddings)
        self.sequences = sequences
        self.metadata = metadata
        print(f"Індекс: {len(metadata)} треків, {embeddings.shape[1]} фіч")

    def search(self, query_emb: np.ndarray, query_seq: np.ndarray = None) -> tuple:
        """
        Пошук найближчого.

        Якщо use_dtw=True: FAISS → кандидати → DTW
        Якщо use_dtw=False: тільки FAISS
        """
        query_emb = query_emb.reshape(1, -1).astype("float32")

        if not self.use_dtw:
            # Тільки FAISS
            scores, indices = self.faiss_index.search(query_emb, 1)
            return self.metadata[indices[0][0]], float(scores[0][0])

        # Гібрид: FAISS + DTW
        n_search = min(self.n_candidates, len(self.sequences))
        scores, indices = self.faiss_index.search(query_emb, n_search)

        best_idx, best_dist = None, float("inf")
        for idx in indices[0]:
            dist = dtw_distance(query_seq, self.sequences[idx])
            if dist < best_dist:
                best_dist = dist
                best_idx = idx

        similarity = 1.0 / (1.0 + best_dist)
        return self.metadata[best_idx], similarity


# ============================================================
# API + КЕШ
# ============================================================

def analyze_file(filepath: Path, api_url: str, timeout: int) -> dict | None:
    """Запит до API."""
    try:
        with open(filepath, "rb") as f:
            files = {"file": (filepath.name, f, "audio/wav")}
            response = requests.post(api_url, files=files, timeout=timeout)

        if response.status_code != 200:
            print(f"[HTTP {response.status_code}] {filepath.name}")
            return None

        return parse_api_response(response.json())
    except Exception as e:
        print(f"[ERROR] {filepath.name}: {e}")
        return None


def cache_path(cache_dir: Path, filename: str) -> Path:
    return cache_dir / f"{filename}.npz"


def save_cache(cache_dir: Path, filename: str, data: dict):
    cache_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_path(cache_dir, filename),
        matrix=data["matrix"],
        structure=data["structure"],
    )


def load_cache(cache_dir: Path, filename: str) -> dict | None:
    path = cache_path(cache_dir, filename)
    if not path.exists():
        return None
    try:
        loaded = np.load(path, allow_pickle=True)
        return {"matrix": loaded["matrix"], "structure": str(loaded["structure"])}
    except:
        return None


# ============================================================
# ОБРОБКА
# ============================================================

def find_wav_files(folder: Path) -> list[Path]:
    if not Path(folder).exists():
        return []
    return sorted(p for p in Path(folder).rglob("*.wav") if p.is_file())


def ensure_wav(name: str) -> str:
    name = str(name)
    return name if name.lower().endswith(".wav") else name + ".wav"


def process_folder(folder: Path, cfg: Config) -> tuple:
    """Обробляє папку: API → rich embedding."""
    files = find_wav_files(folder)
    print(f"Файлів: {len(files)}")

    if not files:
        raise RuntimeError(f"Не знайдено .wav у {folder}")

    embeddings, sequences, metadata = [], [], []

    def process_one(filepath: Path):
        if cfg.use_cache:
            cached = load_cache(cfg.cache_dir, filepath.name)
            if cached:
                return filepath, cached

        result = analyze_file(filepath, cfg.api_url, cfg.api_timeout)

        if result and cfg.use_cache:
            save_cache(cfg.cache_dir, filepath.name, result)

        return filepath, result

    with ThreadPoolExecutor(max_workers=cfg.max_workers) as executor:
        futures = {executor.submit(process_one, f): f for f in files}

        for future in tqdm(as_completed(futures), total=len(files), desc="API"):
            filepath, result = future.result()

            if result is None:
                continue

            matrix = result["matrix"]
            emb = compute_rich_embedding(matrix, cfg.n_parts)

            embeddings.append(emb)
            sequences.append(matrix)
            metadata.append({
                "name": filepath.name,
                "path": str(filepath),
                "structure": result.get("structure", ""),
            })

    if not embeddings:
        raise RuntimeError("Не вдалось обробити жодного файлу")

    print(f"Успішно: {len(embeddings)}")
    return np.stack(embeddings), sequences, metadata


# ============================================================
# ГОЛОВНА
# ============================================================

def run(cfg: Config):
    # Перевірка
    ori_files = find_wav_files(cfg.origin_dir)
    comp_files = find_wav_files(cfg.comparison_dir)

    print("=" * 50)
    print("ШЛЯХИ")
    print("=" * 50)
    print(f"Origin: {cfg.origin_dir} ({len(ori_files)} файлів)")
    print(f"Comparison: {cfg.comparison_dir} ({len(comp_files)} файлів)")
    print(f"CSV: {cfg.pairs_csv}")

    if not ori_files or not comp_files:
        print("\n❌ Не знайдено файлів!")
        return None

    print("\n" + "=" * 50)
    print("НАЛАШТУВАННЯ")
    print("=" * 50)
    print(f"Частин треку: {cfg.n_parts}")
    print(f"DTW: {'✓' if cfg.use_dtw else '✗'}")
    print(f"FAISS кандидатів: {cfg.faiss_candidates}")

    # 1. Comparison
    print("\n" + "=" * 50)
    print("[1/4] ОБРОБКА COMPARISON")
    print("=" * 50)
    embeddings, sequences, metadata = process_folder(cfg.comparison_dir, cfg)

    # Інфо про ембединг
    print(f"\nРозмір ембедингу: {embeddings.shape[1]}")

    # 2. Індекс
    print("\n" + "=" * 50)
    print("[2/4] БУДУЮ ІНДЕКС")
    print("=" * 50)
    index = SearchIndex(use_dtw=cfg.use_dtw, n_candidates=cfg.faiss_candidates)
    index.build(embeddings, sequences, metadata)

    # 3. CSV
    print("\n" + "=" * 50)
    print("[3/4] ЧИТАЮ ПАРИ")
    print("=" * 50)
    df = pd.read_csv(cfg.pairs_csv)
    print(f"Пар: {len(df)}")
    origin_map = {p.name: p for p in ori_files}

    # 4. Пошук
    print("\n" + "=" * 50)
    print("[4/4] ПОШУК")
    print("=" * 50)

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Пошук"):
        ori_name = ensure_wav(row["ori_title"])
        comp_name = ensure_wav(row["comp_title"])

        ori_path = origin_map.get(ori_name)

        if ori_path is None:
            results.append({
                "ori": ori_name,
                "expected": comp_name,
                "predicted": None,
                "score": None,
                "match": False,
                "status": "ORI_NOT_FOUND"
            })
            continue

        # Отримуємо дані
        if cfg.use_cache:
            cached = load_cache(cfg.cache_dir, ori_name)
            if cached:
                matrix = cached["matrix"]
            else:
                result = analyze_file(ori_path, cfg.api_url, cfg.api_timeout)
                if result is None:
                    results.append({
                        "ori": ori_name,
                        "expected": comp_name,
                        "predicted": None,
                        "score": None,
                        "match": False,
                        "status": "API_ERROR"
                    })
                    continue
                save_cache(cfg.cache_dir, ori_name, result)
                matrix = result["matrix"]
        else:
            result = analyze_file(ori_path, cfg.api_url, cfg.api_timeout)
            if result is None:
                results.append({
                    "ori": ori_name,
                    "expected": comp_name,
                    "predicted": None,
                    "score": None,
                    "match": False,
                    "status": "API_ERROR"
                })
                continue
            matrix = result["matrix"]

        # Ембединг + пошук
        query_emb = compute_rich_embedding(matrix, cfg.n_parts)
        query_seq = matrix if cfg.use_dtw else None

        match, score = index.search(query_emb, query_seq)

        results.append({
            "ori": ori_name,
            "expected": comp_name,
            "predicted": match["name"],
            "score": round(score, 4),
            "match": match["name"] == comp_name,
            "ori_structure": "",  # можна додати
            "pred_structure": match.get("structure", ""),
            "status": "OK"
        })

    # Результати
    results_df = pd.DataFrame(results)
    valid = results_df[results_df["status"] == "OK"]
    accuracy = valid["match"].mean() if len(valid) > 0 else 0

    print("\n" + "=" * 50)
    print("РЕЗУЛЬТАТИ")
    print("=" * 50)
    print(f"Всього: {len(results_df)}")
    print(f"Перевірено: {len(valid)}")
    print(f"Точність: {accuracy:.2%}")

    print(f"\nСтатуси:")
    print(results_df["status"].value_counts().to_string())

    if len(valid) > 0:
        print(f"\nРозподіл scores:")
        print(f"  min:  {valid['score'].min():.4f}")
        print(f"  mean: {valid['score'].mean():.4f}")
        print(f"  max:  {valid['score'].max():.4f}")

    errors = valid[~valid["match"]].sort_values("score", ascending=False)
    if len(errors) > 0:
        print(f"\nПомилки ({len(errors)}):")
        print(errors[["ori", "expected", "predicted", "score"]].head(10).to_string(index=False))

    # Зберігаємо
    output = Path("./results_rich.csv")
    results_df.to_csv(output, index=False)
    print(f"\nЗбережено: {output}")

    return results_df


# ============================================================
# ЕКСПЕРИМЕНТИ
# ============================================================

def run_experiments(cfg: Config):
    """Порівнює різні налаштування."""

    experiments = [
        {"n_parts": 4, "use_dtw": False, "name": "rich_only"},
        {"n_parts": 4, "use_dtw": True, "faiss_candidates": 20, "name": "rich+dtw_20"},
        {"n_parts": 4, "use_dtw": True, "faiss_candidates": 50, "name": "rich+dtw_50"},
        {"n_parts": 8, "use_dtw": False, "name": "rich_8parts"},
        {"n_parts": 2, "use_dtw": False, "name": "rich_2parts"},
    ]

    results_summary = []

    for exp in experiments:
        print(f"\n{'='*60}")
        print(f"ЕКСПЕРИМЕНТ: {exp['name']}")
        print(f"{'='*60}")

        cfg.n_parts = exp.get("n_parts", 4)
        cfg.use_dtw = exp.get("use_dtw", False)
        cfg.faiss_candidates = exp.get("faiss_candidates", 30)

        try:
            results = run(cfg)
            valid = results[results["status"] == "OK"]
            accuracy = valid["match"].mean() if len(valid) > 0 else 0

            results_summary.append({
                "experiment": exp["name"],
                "n_parts": cfg.n_parts,
                "use_dtw": cfg.use_dtw,
                "candidates": cfg.faiss_candidates,
                "accuracy": f"{accuracy:.2%}",
            })
        except Exception as e:
            print(f"Помилка: {e}")
            results_summary.append({
                "experiment": exp["name"],
                "error": str(e),
            })

    print("\n" + "=" * 60)
    print("ПОРІВНЯННЯ ЕКСПЕРИМЕНТІВ")
    print("=" * 60)
    print(pd.DataFrame(results_summary).to_string(index=False))


cfg = Config()

run(cfg)


ШЛЯХИ
Origin: /content/original (19 файлів)
Comparison: /content/comparison (16 файлів)
CSV: /content/song_pairs.csv

НАЛАШТУВАННЯ
Частин треку: 4
DTW: ✓
FAISS кандидатів: 30

[1/4] ОБРОБКА COMPARISON
Файлів: 16


API: 100%|██████████| 16/16 [00:00<00:00, 451.03it/s]


Успішно: 16

Розмір ембедингу: 136

[2/4] БУДУЮ ІНДЕКС
Індекс: 16 треків, 136 фіч

[3/4] ЧИТАЮ ПАРИ
Пар: 16

[4/4] ПОШУК


Пошук: 100%|██████████| 16/16 [00:06<00:00,  2.36it/s]


РЕЗУЛЬТАТИ
Всього: 16
Перевірено: 15
Точність: 13.33%

Статуси:
status
OK               15
ORI_NOT_FOUND     1

Розподіл scores:
  min:  0.9573
  mean: 0.9829
  max:  0.9964

Помилки (13):
                                                                             ori                                                                         expected                                                           predicted  score
                                     Queen - Under Pressure _Official Video_.wav                            Vanilla Ice - Ice Ice Baby _Official Music Video_.wav                            Smoke On The Water _2024 Remastered_.wav 0.9964
                                        The Gap Band - Oops Upside Your Head.wav                    Mark Ronson - Uptown Funk _Official Video_ ft_ Bruno Mars.wav                            Smoke On The Water _2024 Remastered_.wav 0.9939
                    Jennifer Rush - The Power Of Love _Official Video_ _VOD_.wav               Céli

,ori,expected,predicted,score,match,ori_structure,pred_structure,status
0,The Gap Band - Oops Upside Your Head.wav,Mark Ronson - Uptown Funk _Official Video_ ft_...,Smoke On The Water _2024 Remastered_.wav,0.9939,False,,chorus-inst,OK
1,Jennifer Rush - The Power Of Love _Official Vi...,Céline Dion - The Power Of Love _Official Rema...,Céline Dion - The Power Of Love _Official Rem...,0.9902,False,,verse-chorus,OK
2,Without you - Badfinger.wav,Mariah Carey - Without You _Official Lyric Vid...,Mariah Carey - Without You _Official Lyric Vid...,0.9919,True,,verse-chorus,OK
3,_Official Audio_ 이정현_Lee Jung-hyun_ - 와.wav,Bandido - Vamos Amigos _Eurodance Version_.wav,None,NaN,False,NaN,NaN,ORI_NOT_FOUND
4,Queen - Under Pressure _Official Video_.wav,Vanilla Ice - Ice Ice Baby _Official Music Vid...,Smoke On The Water _2024 Remastered_.wav,0.9964,False,,chorus-inst,OK
5,Joyful Noise - Official Music Video_ FLAME fea...,Katy Perry - Dark Horse _Lyrics_ ft_ Juicy J.wav,Vanilla Ice - Ice Ice Baby _Official Music Vid...,0.9814,False,,intro-verse,OK
6,Astrud Gilberto - Maria Quiet.wav,Smoke On The Water _2024 Remastered_.wav,_DANCE_ 싸이 _PSY_ - 챔피언.wav,0.9870,False,,chorus-verse,OK
7,Crazy Frog - Axel F _Official Video_.wav,_DANCE_ 싸이 _PSY_ - 챔피언.wav,Mark Ronson - Uptown Funk _Official Video_ ft_...,0.9786,False,,chorus-verse,OK
8,Sting - Shape of My Heart _Official Music Vide...,Juice WRLD - Lucid Dreams _Official Music Vide...,Juice WRLD - Lucid Dreams _Official Music Vide...,0.9958,True,,intro-verse,OK
9,September - Cry For You.wav,Ed Sheeran - Bad Habits _Official Lyric Video_...,_DANCE_ 싸이 _PSY_ - 챔피언.wav,0.9661,False,,chorus-verse,OK


In [ ]:
"""
Пошук схожих пісень: Structure API + CLAP audio embeddings.

Комбінує два типи ембедингів:
1. Structure embedding (з API) — структура пісні (verse/chorus/etc)
2. CLAP embedding — аудіо контент (мелодія, тембр, ритм)

Запуск:
    pip install requests numpy pandas faiss-cpu scipy tqdm transformers torch librosa
    python combined_search.py
"""

import json
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn.functional as F
import faiss
from scipy.spatial.distance import cdist
from tqdm import tqdm
from transformers import ClapModel, ClapProcessor


# ============================================================
# НАЛАШТУВАННЯ
# ============================================================

@dataclass
class Config:
    # API структури
    api_url: str = "http://13.220.223.194:8000/analyze"
    api_timeout: int = 300

    # CLAP
    clap_model: str = "laion/larger_clap_music"
    clap_sample_rate: int = 48000
    clap_chunk_sec: float = 10.0
    clap_hop_sec: float = 5.0

    # Шляхи
    origin_dir: Path = Path("/content/original")
    comparison_dir: Path = Path("/content/comparison")
    pairs_csv: Path = Path("/content/song_pairs.csv")
    cache_dir: Path = Path("./cache_combined")

    # Ембединг
    n_parts: int = 4
    use_dtw: bool = True
    faiss_candidates: int = 1

    # Ваги комбінації
    weight_structure: float = 0.5  # вага structure embedding
    weight_clap: float = 0.5       # вага CLAP embedding

    # Інше
    max_workers: int = 4
    use_cache: bool = True


# ============================================================
# CLAP ENCODER
# ============================================================

class CLAPEncoder:
    """Кодує аудіо через CLAP модель."""

    def __init__(self, model_id: str):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"CLAP device: {self.device}")
        print(f"Loading CLAP: {model_id}")

        self.model = ClapModel.from_pretrained(model_id).to(self.device).eval()
        self.processor = ClapProcessor.from_pretrained(model_id)
        self.dim = 512  # CLAP embedding dimension

    def load_audio(self, path: Path, sr: int) -> np.ndarray:
        """Завантажує аудіо."""
        audio, _ = librosa.load(str(path), sr=sr, mono=True)
        return np.nan_to_num(audio.astype(np.float32))

    @torch.no_grad()
    def encode_chunk(self, chunk: np.ndarray, sr: int) -> np.ndarray:
        """Один чанк → ембединг (512,)."""
        inputs = self.processor(audios=chunk, sampling_rate=sr, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        emb = self.model.get_audio_features(**inputs)
        emb = F.normalize(emb, p=2, dim=-1)

        return emb.cpu().numpy().squeeze().astype("float32")

    def encode_file(self, path: Path, sr: int, chunk_sec: float, hop_sec: float) -> dict:
        """
        Файл → CLAP ембединги.

        Повертає:
        - sequence: (N, 512) — послідовність ембедингів чанків
        - mean: (512,) — усереднений
        """
        audio = self.load_audio(path, sr)

        chunk_len = int(chunk_sec * sr)
        hop_len = int(hop_sec * sr)

        # Розбиваємо на чанки
        if len(audio) <= chunk_len:
            chunks = [audio]
        else:
            chunks = []
            for start in range(0, len(audio) - chunk_len + 1, hop_len):
                chunks.append(audio[start:start + chunk_len])
            if not chunks:
                chunks = [audio]

        # Кодуємо кожен чанк
        embeddings = [self.encode_chunk(ch, sr) for ch in chunks]
        sequence = np.stack(embeddings)

        # Усереднюємо
        mean = sequence.mean(axis=0)
        mean = mean / (np.linalg.norm(mean) + 1e-12)

        return {
            "sequence": sequence,
            "mean": mean,
        }


# ============================================================
# STRUCTURE API
# ============================================================

def parse_structure_response(data: dict) -> dict | None:
    """Парсить відповідь Structure API."""
    logits = data.get("logits", {})

    boundary = logits.get("boundary_logits", [])
    if not boundary:
        return None

    T = len(boundary)
    boundary = np.array(boundary, dtype=np.float32).reshape(T, 1)

    functions = logits.get("function_logits_topk", [])
    if not functions:
        return None

    func_matrix = []
    for func in functions:
        values = func.get("values", [])
        if len(values) == T:
            func_matrix.append(values)

    if not func_matrix:
        return None

    func_matrix = np.array(func_matrix, dtype=np.float32).T
    matrix = np.hstack([boundary, func_matrix])

    return {
        "matrix": matrix,
        "structure": data.get("structure", ""),
    }


def call_structure_api(filepath: Path, api_url: str, timeout: int) -> dict | None:
    """Запит до Structure API."""
    try:
        with open(filepath, "rb") as f:
            files = {"file": (filepath.name, f, "audio/wav")}
            response = requests.post(api_url, files=files, timeout=timeout)

        if response.status_code != 200:
            return None

        return parse_structure_response(response.json())
    except:
        return None


# ============================================================
# STRUCTURE EMBEDDING (RICH)
# ============================================================

def compute_structure_embedding(matrix: np.ndarray, n_parts: int = 4) -> np.ndarray:
    """
    Багатий ембединг структури пісні.
    """
    T, D = matrix.shape
    part_size = T // n_parts
    features = []

    # 1. Позиційні mean
    for i in range(n_parts):
        start = i * part_size
        end = start + part_size if i < n_parts - 1 else T
        features.append(matrix[start:end].mean(axis=0))

    # 2. Статистики
    features.append(matrix.std(axis=0))
    features.append(matrix.max(axis=0))
    features.append(matrix.min(axis=0))

    # 3. Позиції максимумів
    features.append(matrix.argmax(axis=0) / T)

    # 4. Тренд
    first_q = matrix[:T//4].mean(axis=0)
    last_q = matrix[-T//4:].mean(axis=0)
    features.append(last_q - first_q)

    # 5. Активність
    diffs = np.abs(np.diff(matrix, axis=0))
    features.append(diffs.mean(axis=0))

    change_mag = diffs.sum(axis=1)
    hist, _ = np.histogram(np.arange(len(change_mag)), bins=n_parts, weights=change_mag)
    hist = hist / (hist.sum() + 1e-12)
    features.append(hist)

    # 6. Boundary peaks
    boundary = matrix[:, 0]
    top_k = min(5, len(boundary))
    features.append(np.sort(boundary)[-top_k:])
    features.append(np.sort(np.argsort(boundary)[-top_k:]) / T)

    # 7. Домінантні функції
    n_funcs = D - 1
    for i in range(n_parts):
        start = i * part_size
        end = start + part_size if i < n_parts - 1 else T
        func_means = matrix[start:end, 1:].mean(axis=0)
        one_hot = np.zeros(n_funcs)
        one_hot[func_means.argmax()] = 1
        features.append(one_hot)

    emb = np.concatenate(features)
    return (emb / (np.linalg.norm(emb) + 1e-12)).astype("float32")


# ============================================================
# КОМБІНОВАНИЙ ЕМБЕДИНГ
# ============================================================

def compute_combined_embedding(
    structure_emb: np.ndarray,
    clap_emb: np.ndarray,
    weight_structure: float,
    weight_clap: float
) -> np.ndarray:
    """
    Комбінує structure і CLAP ембединги.

    Варіанти комбінації:
    1. Конкатенація (зберігає всю інформацію)
    2. Зважена сума (якщо однаковий розмір)
    """
    # Нормалізуємо
    structure_emb = structure_emb / (np.linalg.norm(structure_emb) + 1e-12)
    clap_emb = clap_emb / (np.linalg.norm(clap_emb) + 1e-12)

    # Зважуємо
    structure_emb = structure_emb * weight_structure
    clap_emb = clap_emb * weight_clap

    # Конкатенуємо
    combined = np.concatenate([structure_emb, clap_emb])

    # Фінальна нормалізація
    combined = combined / (np.linalg.norm(combined) + 1e-12)

    return combined.astype("float32")


# ============================================================
# DTW
# ============================================================

def dtw_distance(seq1: np.ndarray, seq2: np.ndarray) -> float:
    """DTW для послідовностей."""
    # Зменшуємо для швидкості
    step = max(1, min(len(seq1), len(seq2)) // 100)
    seq1 = seq1[::step]
    seq2 = seq2[::step]

    n, m = len(seq1), len(seq2)
    cost = cdist(seq1, seq2, metric="cosine")

    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i, j] = cost[i-1, j-1] + min(dp[i-1, j], dp[i, j-1], dp[i-1, j-1])

    return dp[n, m] / (n + m)


def combined_dtw_distance(
    struct_seq1: np.ndarray, struct_seq2: np.ndarray,
    clap_seq1: np.ndarray, clap_seq2: np.ndarray,
    weight_structure: float, weight_clap: float
) -> float:
    """Комбінована DTW відстань."""
    dist_struct = dtw_distance(struct_seq1, struct_seq2)
    dist_clap = dtw_distance(clap_seq1, clap_seq2)

    return weight_structure * dist_struct + weight_clap * dist_clap


# ============================================================
# ПОШУКОВИЙ ІНДЕКС
# ============================================================

class CombinedSearchIndex:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.faiss_index = None
        self.struct_sequences = []
        self.clap_sequences = []
        self.metadata = []

    def build(self, embeddings: np.ndarray, struct_seqs: list, clap_seqs: list, metadata: list):
        self.faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
        self.faiss_index.add(embeddings)
        self.struct_sequences = struct_seqs
        self.clap_sequences = clap_seqs
        self.metadata = metadata
        print(f"Індекс: {len(metadata)} треків, {embeddings.shape[1]} фіч")

    def search(self, query_emb: np.ndarray,
               query_struct_seq: np.ndarray = None,
               query_clap_seq: np.ndarray = None) -> tuple:
        """Пошук з опціональним DTW."""
        query_emb = query_emb.reshape(1, -1).astype("float32")

        if not self.cfg.use_dtw:
            scores, indices = self.faiss_index.search(query_emb, 1)
            return self.metadata[indices[0][0]], float(scores[0][0])

        # FAISS → кандидати → DTW
        n_search = min(self.cfg.faiss_candidates, len(self.metadata))
        _, indices = self.faiss_index.search(query_emb, n_search)

        best_idx, best_dist = None, float("inf")

        for idx in indices[0]:
            dist = combined_dtw_distance(
                query_struct_seq, self.struct_sequences[idx],
                query_clap_seq, self.clap_sequences[idx],
                self.cfg.weight_structure, self.cfg.weight_clap
            )
            if dist < best_dist:
                best_dist = dist
                best_idx = idx

        return self.metadata[best_idx], 1.0 / (1.0 + best_dist)


# ============================================================
# КЕШ
# ============================================================

def cache_path(cache_dir: Path, filename: str) -> Path:
    return cache_dir / f"{filename}.npz"


def save_cache(cache_dir: Path, filename: str, data: dict):
    cache_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_path(cache_dir, filename),
        struct_matrix=data["struct_matrix"],
        clap_sequence=data["clap_sequence"],
        clap_mean=data["clap_mean"],
        structure=data.get("structure", ""),
    )


def load_cache(cache_dir: Path, filename: str) -> dict | None:
    path = cache_path(cache_dir, filename)
    if not path.exists():
        return None
    try:
        loaded = np.load(path, allow_pickle=True)
        return {
            "struct_matrix": loaded["struct_matrix"],
            "clap_sequence": loaded["clap_sequence"],
            "clap_mean": loaded["clap_mean"],
            "structure": str(loaded["structure"]),
        }
    except:
        return None


# ============================================================
# ОБРОБКА
# ============================================================

def find_wav_files(folder: Path) -> list[Path]:
    if not Path(folder).exists():
        return []
    return sorted(p for p in Path(folder).rglob("*.wav") if p.is_file())


def ensure_wav(name: str) -> str:
    name = str(name)
    return name if name.lower().endswith(".wav") else name + ".wav"


def process_file(filepath: Path, cfg: Config, clap_encoder: CLAPEncoder) -> dict | None:
    """Обробляє один файл: Structure API + CLAP."""

    # Structure API
    struct_data = call_structure_api(filepath, cfg.api_url, cfg.api_timeout)
    if struct_data is None:
        return None

    # CLAP
    try:
        clap_data = clap_encoder.encode_file(
            filepath, cfg.clap_sample_rate,
            cfg.clap_chunk_sec, cfg.clap_hop_sec
        )
    except Exception as e:
        print(f"[CLAP ERROR] {filepath.name}: {e}")
        return None

    return {
        "struct_matrix": struct_data["matrix"],
        "clap_sequence": clap_data["sequence"],
        "clap_mean": clap_data["mean"],
        "structure": struct_data.get("structure", ""),
    }


def process_folder(folder: Path, cfg: Config, clap_encoder: CLAPEncoder) -> tuple:
    """Обробляє папку."""
    files = find_wav_files(folder)
    print(f"Файлів: {len(files)}")

    if not files:
        raise RuntimeError(f"Не знайдено .wav у {folder}")

    embeddings = []
    struct_sequences = []
    clap_sequences = []
    metadata = []

    for filepath in tqdm(files, desc="Обробка"):
        # Кеш
        if cfg.use_cache:
            cached = load_cache(cfg.cache_dir, filepath.name)
            if cached:
                data = cached
            else:
                data = process_file(filepath, cfg, clap_encoder)
                if data:
                    save_cache(cfg.cache_dir, filepath.name, data)
        else:
            data = process_file(filepath, cfg, clap_encoder)

        if data is None:
            print(f"[SKIP] {filepath.name}")
            continue

        # Ембединги
        struct_emb = compute_structure_embedding(data["struct_matrix"], cfg.n_parts)
        combined_emb = compute_combined_embedding(
            struct_emb, data["clap_mean"],
            cfg.weight_structure, cfg.weight_clap
        )

        embeddings.append(combined_emb)
        struct_sequences.append(data["struct_matrix"])
        clap_sequences.append(data["clap_sequence"])
        metadata.append({
            "name": filepath.name,
            "path": str(filepath),
            "structure": data.get("structure", ""),
        })

    if not embeddings:
        raise RuntimeError("Не вдалось обробити жодного файлу")

    print(f"Успішно: {len(embeddings)}")
    return np.stack(embeddings), struct_sequences, clap_sequences, metadata


# ============================================================
# ГОЛОВНА
# ============================================================

def run(cfg: Config):
    ori_files = find_wav_files(cfg.origin_dir)
    comp_files = find_wav_files(cfg.comparison_dir)

    print("=" * 60)
    print("COMBINED SEARCH: Structure + CLAP")
    print("=" * 60)
    print(f"Origin: {cfg.origin_dir} ({len(ori_files)} файлів)")
    print(f"Comparison: {cfg.comparison_dir} ({len(comp_files)} файлів)")
    print(f"\nНалаштування:")
    print(f"  Structure weight: {cfg.weight_structure}")
    print(f"  CLAP weight: {cfg.weight_clap}")
    print(f"  DTW: {'✓' if cfg.use_dtw else '✗'}")
    print(f"  n_parts: {cfg.n_parts}")

    if not ori_files or not comp_files:
        print("\n❌ Не знайдено файлів!")
        return None

    # CLAP encoder
    print("\n" + "=" * 60)
    print("[1/5] ЗАВАНТАЖЕННЯ CLAP")
    print("=" * 60)
    clap_encoder = CLAPEncoder(cfg.clap_model)

    # Comparison
    print("\n" + "=" * 60)
    print("[2/5] ОБРОБКА COMPARISON")
    print("=" * 60)
    embeddings, struct_seqs, clap_seqs, metadata = process_folder(
        cfg.comparison_dir, cfg, clap_encoder
    )

    # Індекс
    print("\n" + "=" * 60)
    print("[3/5] БУДУЮ ІНДЕКС")
    print("=" * 60)
    index = CombinedSearchIndex(cfg)
    index.build(embeddings, struct_seqs, clap_seqs, metadata)

    # CSV
    print("\n" + "=" * 60)
    print("[4/5] ЧИТАЮ ПАРИ")
    print("=" * 60)
    df = pd.read_csv(cfg.pairs_csv)
    print(f"Пар: {len(df)}")
    origin_map = {p.name: p for p in ori_files}

    # Пошук
    print("\n" + "=" * 60)
    print("[5/5] ПОШУК")
    print("=" * 60)

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Пошук"):
        ori_name = ensure_wav(row["ori_title"])
        comp_name = ensure_wav(row["comp_title"])

        ori_path = origin_map.get(ori_name)

        if ori_path is None:
            results.append({
                "ori": ori_name,
                "expected": comp_name,
                "predicted": None,
                "score": None,
                "match": False,
                "status": "ORI_NOT_FOUND"
            })
            continue

        # Отримуємо дані
        if cfg.use_cache:
            cached = load_cache(cfg.cache_dir, ori_name)
            if cached:
                data = cached
            else:
                data = process_file(ori_path, cfg, clap_encoder)
                if data:
                    save_cache(cfg.cache_dir, ori_name, data)
        else:
            data = process_file(ori_path, cfg, clap_encoder)

        if data is None:
            results.append({
                "ori": ori_name,
                "expected": comp_name,
                "predicted": None,
                "score": None,
                "match": False,
                "status": "PROCESS_ERROR"
            })
            continue

        # Ембединг
        struct_emb = compute_structure_embedding(data["struct_matrix"], cfg.n_parts)
        query_emb = compute_combined_embedding(
            struct_emb, data["clap_mean"],
            cfg.weight_structure, cfg.weight_clap
        )

        # Пошук
        if cfg.use_dtw:
            match, score = index.search(
                query_emb, data["struct_matrix"], data["clap_sequence"]
            )
        else:
            match, score = index.search(query_emb)

        results.append({
            "ori": ori_name,
            "expected": comp_name,
            "predicted": match["name"],
            "score": round(score, 4),
            "match": match["name"] == comp_name,
            "status": "OK"
        })

    # Результати
    results_df = pd.DataFrame(results)
    valid = results_df[results_df["status"] == "OK"]
    accuracy = valid["match"].mean() if len(valid) > 0 else 0

    print("\n" + "=" * 60)
    print("РЕЗУЛЬТАТИ")
    print("=" * 60)
    print(f"Всього: {len(results_df)}")
    print(f"Перевірено: {len(valid)}")
    print(f"Точність: {accuracy:.2%}")

    print(f"\nСтатуси:")
    print(results_df["status"].value_counts().to_string())

    if len(valid) > 0:
        print(f"\nScores:")
        print(f"  min:  {valid['score'].min():.4f}")
        print(f"  mean: {valid['score'].mean():.4f}")
        print(f"  max:  {valid['score'].max():.4f}")

    errors = valid[~valid["match"]].sort_values("score", ascending=False)
    if len(errors) > 0:
        print(f"\nПомилки ({len(errors)}):")
        print(errors[["ori", "expected", "predicted", "score"]].head(10).to_string(index=False))

    output = Path("./results_combined.csv")
    results_df.to_csv(output, index=False)
    print(f"\nЗбережено: {output}")

    return results_df


# ============================================================
# ЕКСПЕРИМЕНТИ З ВАГАМИ
# ============================================================

def run_weight_experiments(cfg: Config):
    """Порівнює різні комбінації ваг."""

    weight_configs = [
        (1.0, 0.0, "structure_only"),
        (0.0, 1.0, "clap_only"),
        (0.7, 0.3, "struct_70_clap_30"),
        (0.5, 0.5, "struct_50_clap_50"),
        (0.3, 0.7, "struct_30_clap_70"),
    ]

    results_summary = []

    for w_struct, w_clap, name in weight_configs:
        print(f"\n{'='*60}")
        print(f"ЕКСПЕРИМЕНТ: {name}")
        print(f"{'='*60}")

        cfg.weight_structure = w_struct
        cfg.weight_clap = w_clap

        try:
            results = run(cfg)
            valid = results[results["status"] == "OK"]
            accuracy = valid["match"].mean() if len(valid) > 0 else 0

            results_summary.append({
                "experiment": name,
                "weight_structure": w_struct,
                "weight_clap": w_clap,
                "accuracy": f"{accuracy:.2%}",
            })
        except Exception as e:
            print(f"Помилка: {e}")

    print("\n" + "=" * 60)
    print("ПОРІВНЯННЯ")
    print("=" * 60)
    print(pd.DataFrame(results_summary).to_string(index=False))


cfg = Config()

    # Налаштуй шляхи:
cfg.origin_dir = Path("/content/original")
cfg.comparison_dir = Path("/content/comparison")

    # Налаштуй ваги:
cfg.weight_structure = 0.5  # структура
cfg.weight_clap = 0.5       # аудіо контент

    # Один запуск:
run(cfg)

    # Або порівняти ваги:
    # run_weight_experiments(cfg)

COMBINED SEARCH: Structure + CLAP
Origin: /content/original (19 файлів)
Comparison: /content/comparison (16 файлів)

Налаштування:
  Structure weight: 0.5
  CLAP weight: 0.5
  DTW: ✓
  n_parts: 4

[1/5] ЗАВАНТАЖЕННЯ CLAP
CLAP device: cpu
Loading CLAP: laion/larger_clap_music

[2/5] ОБРОБКА COMPARISON
Файлів: 16


Обробка: 100%|██████████| 16/16 [00:00<00:00, 477.90it/s]


Успішно: 16

[3/5] БУДУЮ ІНДЕКС
Індекс: 16 треків, 648 фіч

[4/5] ЧИТАЮ ПАРИ
Пар: 16

[5/5] ПОШУК


Пошук:  31%|███▏      | 5/16 [00:00<00:00, 42.44it/s]/tmp/ipython-input-3359103017.py:90: FutureWarning: `audios` is deprecated and will be removed in version v4.59.0 for `ClapProcessor.__call__`. Use `audio` instead.
  inputs = self.processor(audios=chunk, sampling_rate=sr, return_tensors="pt")
Пошук: 100%|██████████| 16/16 [01:10<00:00,  4.41s/it]


РЕЗУЛЬТАТИ
Всього: 16
Перевірено: 15
Точність: 13.33%

Статуси:
status
OK               15
ORI_NOT_FOUND     1

Scores:
  min:  0.9697
  mean: 0.9850
  max:  0.9953

Помилки (13):
                                                                             ori                                                                         expected                                                           predicted  score
                    Jennifer Rush - The Power Of Love _Official Video_ _VOD_.wav               Céline Dion - The Power Of Love _Official Remastered HD Video_.wav Céline Dion - The Power Of Love _Official Remastered HD Video_.wav 0.9935
                                     Queen - Under Pressure _Official Video_.wav                            Vanilla Ice - Ice Ice Baby _Official Music Video_.wav                            Smoke On The Water _2024 Remastered_.wav 0.9926
                                               Astrud Gilberto - Maria Quiet.wav                            

,ori,expected,predicted,score,match,status
0,The Gap Band - Oops Upside Your Head.wav,Mark Ronson - Uptown Funk _Official Video_ ft_...,Smoke On The Water _2024 Remastered_.wav,0.9884,False,OK
1,Jennifer Rush - The Power Of Love _Official Vi...,Céline Dion - The Power Of Love _Official Rema...,Céline Dion - The Power Of Love _Official Rem...,0.9935,False,OK
2,Without you - Badfinger.wav,Mariah Carey - Without You _Official Lyric Vid...,Mariah Carey - Without You _Official Lyric Vid...,0.9918,True,OK
3,_Official Audio_ 이정현_Lee Jung-hyun_ - 와.wav,Bandido - Vamos Amigos _Eurodance Version_.wav,None,NaN,False,ORI_NOT_FOUND
4,Queen - Under Pressure _Official Video_.wav,Vanilla Ice - Ice Ice Baby _Official Music Vid...,Smoke On The Water _2024 Remastered_.wav,0.9926,False,OK
5,Joyful Noise - Official Music Video_ FLAME fea...,Katy Perry - Dark Horse _Lyrics_ ft_ Juicy J.wav,Vanilla Ice - Ice Ice Baby _Official Music Vid...,0.9867,False,OK
6,Astrud Gilberto - Maria Quiet.wav,Smoke On The Water _2024 Remastered_.wav,_DANCE_ 싸이 _PSY_ - 챔피언.wav,0.9887,False,OK
7,Crazy Frog - Axel F _Official Video_.wav,_DANCE_ 싸이 _PSY_ - 챔피언.wav,Mark Ronson - Uptown Funk _Official Video_ ft_...,0.9697,False,OK
8,Sting - Shape of My Heart _Official Music Vide...,Juice WRLD - Lucid Dreams _Official Music Vide...,Juice WRLD - Lucid Dreams _Official Music Vide...,0.9953,True,OK
9,September - Cry For You.wav,Ed Sheeran - Bad Habits _Official Lyric Video_...,Mark Ronson - Uptown Funk _Official Video_ ft_...,0.9755,False,OK


In [ ]:
import requests
from pathlib import Path

BASE_URL = "http://13.220.223.194:9000"   # <-- заміни
ENDPOINT = f"{BASE_URL}/convert"

def midifren_convert(
    audio_path: str,
    sound_type: str = "drums",     # vocals|melody|drums|bass
    return_mode: str = "midi",     # midi|stem|both
    bpm=None,
    quantize=False,
    pitchbend=False,
    onset=None,
    note=None,
    groove=None,
    timeout_sec=1200,
    out_name=None,
):
    data = {
        "sound_type": sound_type,
        "return_mode": return_mode,
        "quantize": str(quantize).lower(),
        "pitchbend": str(pitchbend).lower(),
        "timeout_sec": str(timeout_sec),
    }
    if bpm is not None: data["bpm"] = str(bpm)
    if onset is not None: data["onset"] = str(onset)
    if note is not None: data["note"] = str(note)
    if groove is not None: data["groove"] = str(groove)

    with open(audio_path, "rb") as f:
        files_ = {"file": (Path(audio_path).name, f, "application/octet-stream")}
        r = requests.post(ENDPOINT, data=data, files=files_, timeout=timeout_sec)

    if r.status_code != 200:
        raise RuntimeError(f"API error {r.status_code}:\n{r.text[:2000]}")

    # визначаємо ім'я вихідного файлу
    if out_name is None:
        ext = {"midi": "mid", "stem": "wav", "both": "zip"}[return_mode]
        out_name = f"{sound_type}.{ext}" if return_mode != "both" else f"{sound_type}_outputs.zip"

    out_path = Path("/content") / out_name
    out_path.write_bytes(r.content)
    return str(out_path)

# приклад: “просто MIDI” для drums
out = midifren_convert('/content/ТНМК _ _Люба_ Люба_.wav', sound_type="drums", return_mode="midi", quantize=True)
out


RuntimeError: API error 500:
Internal Server Error

In [ ]:
import requests

with open("/content/ТНМК _ _Люба_ Люба_.wav", "rb") as f:
    response = requests.post(
        "http://13.220.223.194:9000/process",
        files={"file": f},
        data={"sound_type": "vocals", "convert_midi": "true"}
    )

result = response.json()

result

{'job_id': '57642c96-faba-4f57-b139-421c59c16160',
 'message': 'Готово!',
 'output_files': ['vocals.mid'],
 'download_urls': ['/download/57642c96-faba-4f57-b139-421c59c16160/vocals.mid']}

In [ ]:
for url in result["download_urls"]:
    r = requests.get(f"http://13.220.223.194:9000{url}")
    with open(url.split("/")[-1], "wb") as f:
        f.write(r.content)

In [ ]:


with open("/content/ТНМК _ _Люба_ Люба_.wav", "rb") as f:
    response = requests.post(
        "http://13.220.223.194:9000/process",
        files={"file": f},
        data={
            "sound_type": "vocals",
            "extract_stem": "true",   # <- додай це
            "convert_midi": "true",
        }
    )

result = response.json()
print(result)

for url in result["download_urls"]:
    if url.endswith(".mid"):
        r = requests.get(f"http://13.220.223.194:9000{url}")
        with open("output.mid", "wb") as f:
            f.write(r.content)
        print("Saved: output.mid")

ConnectionError: HTTPConnectionPool(host='13.220.223.194', port=9000): Max retries exceeded with url: /process (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x78cff07b17c0>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [ ]:
!pip install mido
import mido
NOTE_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

def note_name(n):
    return f"{NOTE_NAMES[n % 12]}{n // 12 - 1}"

mid = mido.MidiFile("/content/vocals.mid")

print(f"Тривалість: {mid.length:.1f} сек")
print(f"Тіків на біт: {mid.ticks_per_beat}\n")

for track in mid.tracks:
    print(f"=== {track.name or 'Track'} ===")
    time_ticks = 0
    time_sec = 0
    tempo = 500000  # default 120 BPM

    for msg in track:
        time_ticks += msg.time
        time_sec = mido.tick2second(time_ticks, mid.ticks_per_beat, tempo)

        if msg.type == 'set_tempo':
            tempo = msg.tempo
        elif msg.type == 'note_on' and msg.velocity > 0:
            print(f"{time_sec:6.2f}s | {note_name(msg.note):4} | vel={msg.velocity}")

Тривалість: 57.0 сек
Тіків на біт: 220

=== Track ===
=== Track ===
  0.67s | A3   | vel=64
  1.43s | C4   | vel=87
  3.32s | A#3  | vel=46
  4.64s | A#2  | vel=52
  4.65s | A#1  | vel=61
  4.81s | A#3  | vel=53
  5.19s | A#3  | vel=59
  5.43s | C#4  | vel=52
  7.37s | G#3  | vel=47
  8.11s | F2   | vel=51
  8.44s | F3   | vel=62
  8.67s | A#1  | vel=88
  8.81s | F3   | vel=47
  9.18s | F3   | vel=38
  9.32s | C#2  | vel=67
  9.37s | A#2  | vel=35
  9.86s | F2   | vel=71
 10.10s | D#2  | vel=76
 10.34s | C#2  | vel=66
 10.34s | C#4  | vel=68
 10.59s | A#1  | vel=81
 10.60s | F3   | vel=66
 10.61s | A#2  | vel=67
 11.16s | F3   | vel=46
 11.25s | C#2  | vel=72
 11.52s | A#2  | vel=52
 11.57s | D#2  | vel=68
 11.84s | F2   | vel=74
 12.04s | D#2  | vel=69
 12.34s | C#2  | vel=65
 12.35s | C#4  | vel=67
 12.57s | A#3  | vel=57
 12.59s | A#1  | vel=71
 13.30s | C#4  | vel=63
 13.87s | A#3  | vel=66
 14.82s | A#3  | vel=65
 16.50s | A#1  | vel=79
 16.63s | F3   | vel=51
 17.28s | C#2  | vel

In [ ]:
"""
MIDI Similarity Comparison using N-grams

Порівнює два MIDI файли за подібністю нот використовуючи n-грами.
"""

import mido
from collections import Counter
from typing import List, Tuple


def extract_notes(midi_path: str) -> List[int]:
    """Витягує послідовність нот з MIDI файлу"""
    mid = mido.MidiFile(midi_path)
    notes = []

    for track in mid.tracks:
        for msg in track:
            if msg.type == 'note_on' and msg.velocity > 0:
                notes.append(msg.note)

    return notes


def extract_intervals(notes: List[int]) -> List[int]:
    """Конвертує ноти в інтервали (різниця між сусідніми нотами)"""
    if len(notes) < 2:
        return []
    return [notes[i+1] - notes[i] for i in range(len(notes)-1)]


def get_ngrams(sequence: List[int], n: int) -> List[Tuple]:
    """Створює n-грами з послідовності"""
    if len(sequence) < n:
        return []
    return [tuple(sequence[i:i+n]) for i in range(len(sequence)-n+1)]


def jaccard_similarity(set1: set, set2: set) -> float:
    """Коефіцієнт Жаккара: перетин / об'єднання"""
    if not set1 and not set2:
        return 1.0
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0.0


def cosine_similarity(counter1: Counter, counter2: Counter) -> float:
    """Косинусна подібність між двома Counter"""
    all_keys = set(counter1.keys()) | set(counter2.keys())

    dot_product = sum(counter1.get(k, 0) * counter2.get(k, 0) for k in all_keys)
    norm1 = sum(v**2 for v in counter1.values()) ** 0.5
    norm2 = sum(v**2 for v in counter2.values()) ** 0.5

    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)


def dice_similarity(set1: set, set2: set) -> float:
    """Коефіцієнт Дайса: 2 * перетин / (розмір1 + розмір2)"""
    if not set1 and not set2:
        return 1.0
    intersection = len(set1 & set2)
    return 2 * intersection / (len(set1) + len(set2)) if (len(set1) + len(set2)) > 0 else 0.0


def compare_midi_files(
    midi_path1: str,
    midi_path2: str,
    n_values: List[int] = [2, 3, 4, 5],
    use_intervals: bool = True
) -> dict:
    """
    Порівнює два MIDI файли за n-грамами нот.

    Args:
        midi_path1: шлях до першого MIDI
        midi_path2: шлях до другого MIDI
        n_values: список розмірів n-грам для аналізу
        use_intervals: якщо True - використовує інтервали замість абсолютних нот
                      (краще для пісень в різних тональностях)

    Returns:
        dict з результатами порівняння
    """
    # Витягуємо ноти
    notes1 = extract_notes(midi_path1)
    notes2 = extract_notes(midi_path2)

    # Конвертуємо в інтервали якщо потрібно
    if use_intervals:
        seq1 = extract_intervals(notes1)
        seq2 = extract_intervals(notes2)
        mode = "intervals"
    else:
        seq1 = notes1
        seq2 = notes2
        mode = "notes"

    results = {
        "file1": midi_path1,
        "file2": midi_path2,
        "mode": mode,
        "notes_count": {"file1": len(notes1), "file2": len(notes2)},
        "ngram_analysis": {}
    }

    for n in n_values:
        ngrams1 = get_ngrams(seq1, n)
        ngrams2 = get_ngrams(seq2, n)

        set1 = set(ngrams1)
        set2 = set(ngrams2)

        counter1 = Counter(ngrams1)
        counter2 = Counter(ngrams2)

        results["ngram_analysis"][f"{n}-gram"] = {
            "count": {"file1": len(ngrams1), "file2": len(ngrams2)},
            "unique": {"file1": len(set1), "file2": len(set2)},
            "common": len(set1 & set2),
            "jaccard": round(jaccard_similarity(set1, set2), 4),
            "dice": round(dice_similarity(set1, set2), 4),
            "cosine": round(cosine_similarity(counter1, counter2), 4),
        }

    # Середня подібність
    avg_jaccard = sum(r["jaccard"] for r in results["ngram_analysis"].values()) / len(n_values)
    avg_cosine = sum(r["cosine"] for r in results["ngram_analysis"].values()) / len(n_values)

    results["average_similarity"] = {
        "jaccard": round(avg_jaccard, 4),
        "cosine": round(avg_cosine, 4),
    }

    return results


def print_results(results: dict):
    """Виводить результати в читабельному форматі"""
    print("=" * 60)
    print("MIDI SIMILARITY ANALYSIS")
    print("=" * 60)
    print(f"File 1: {results['file1']}")
    print(f"File 2: {results['file2']}")
    print(f"Mode: {results['mode']}")
    print(f"Notes: {results['notes_count']['file1']} vs {results['notes_count']['file2']}")
    print()

    print("N-gram Analysis:")
    print("-" * 60)
    print(f"{'N-gram':<10} {'Common':<10} {'Jaccard':<10} {'Dice':<10} {'Cosine':<10}")
    print("-" * 60)

    for ngram, data in results["ngram_analysis"].items():
        print(f"{ngram:<10} {data['common']:<10} {data['jaccard']:<10} {data['dice']:<10} {data['cosine']:<10}")

    print("-" * 60)
    print(f"\nAverage Similarity:")
    print(f"  Jaccard: {results['average_similarity']['jaccard']:.2%}")
    print(f"  Cosine:  {results['average_similarity']['cosine']:.2%}")
    print("=" * 60)


# # ============== ВИКОРИСТАННЯ ==============

# if __name__ == "__main__":
#     import sys

#     if len(sys.argv) < 3:
#         print("Використання: python midi_similarity.py <file1.mid> <file2.mid>")
#         print()
#         print("Приклад:")
#         print("  python midi_similarity.py song1.mid song2.mid")
#         sys.exit(1)

#     file1 = sys.argv[1]
#     file2 = sys.argv[2]

#     # Порівняння з інтервалами (краще для різних тональностей)
#     results = compare_midi_files(file1, file2, n_values=[2, 3, 4, 5], use_intervals=True)
#     print_results(results)

#     print("\n\nПорівняння по абсолютних нотах:")
#     results_abs = compare_midi_files(file1, file2, n_values=[2, 3, 4, 5], use_intervals=False)
#     print_results(results_abs)

In [ ]:


results = compare_midi_files("/content/output.mid", "/content/vocals.mid")
print_results(results)


print(results["average_similarity"]["jaccard"])
print(results["average_similarity"]["cosine"])

MIDI SIMILARITY ANALYSIS
File 1: /content/output.mid
File 2: /content/vocals.mid
Mode: intervals
Notes: 35 vs 159

N-gram Analysis:
------------------------------------------------------------
N-gram     Common     Jaccard    Dice       Cosine    
------------------------------------------------------------
2-gram     6          0.0451     0.0863     0.0741    
3-gram     2          0.012      0.0237     0.0237    
4-gram     0          0.0        0.0        0.0       
5-gram     0          0.0        0.0        0.0       
------------------------------------------------------------

Average Similarity:
  Jaccard: 1.43%
  Cosine:  2.44%
0.0143
0.0244


In [ ]:
"""
MIDI Similarity Pipeline

Конвертує WAV файли в MIDI через API, порівнює пари пісень,
знаходить найбільш подібні та рахує точність.

Використання:
    python pipeline.py --api http://your-server:8000
"""

import os
import csv
import requests
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict
import mido

# ============== CONFIG ==============

API_URL = "http://localhost:8000"  # Зміни на свій сервер
ORIGINAL_DIR = "original"
COMPARISON_DIR = "comparison"
MIDI_DIR = "midi_cache"
PAIRS_CSV = "song_pairs.csv"

# ============== MIDI EXTRACTION ==============

def extract_notes(midi_path: str) -> List[int]:
    """Витягує послідовність нот з MIDI файлу"""
    try:
        mid = mido.MidiFile(midi_path)
        notes = []
        for track in mid.tracks:
            for msg in track:
                if msg.type == 'note_on' and msg.velocity > 0:
                    notes.append(msg.note)
        return notes
    except Exception as e:
        print(f"  Помилка читання {midi_path}: {e}")
        return []


def extract_intervals(notes: List[int]) -> List[int]:
    """Конвертує ноти в інтервали"""
    if len(notes) < 2:
        return []
    return [notes[i+1] - notes[i] for i in range(len(notes)-1)]


def get_ngrams(sequence: List[int], n: int) -> List[Tuple]:
    """Створює n-грами"""
    if len(sequence) < n:
        return []
    return [tuple(sequence[i:i+n]) for i in range(len(sequence)-n+1)]


def cosine_similarity(counter1: Counter, counter2: Counter) -> float:
    """Косинусна подібність"""
    all_keys = set(counter1.keys()) | set(counter2.keys())
    dot_product = sum(counter1.get(k, 0) * counter2.get(k, 0) for k in all_keys)
    norm1 = sum(v**2 for v in counter1.values()) ** 0.5
    norm2 = sum(v**2 for v in counter2.values()) ** 0.5
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)


def calculate_similarity(midi1: str, midi2: str, n_values: List[int] = [3, 4, 5]) -> float:
    """Рахує середню подібність між двома MIDI файлами"""
    notes1 = extract_notes(midi1)
    notes2 = extract_notes(midi2)

    if not notes1 or not notes2:
        return 0.0

    # Використовуємо інтервали для інваріантності до тональності
    seq1 = extract_intervals(notes1)
    seq2 = extract_intervals(notes2)

    if not seq1 or not seq2:
        return 0.0

    similarities = []
    for n in n_values:
        ngrams1 = get_ngrams(seq1, n)
        ngrams2 = get_ngrams(seq2, n)

        if ngrams1 and ngrams2:
            counter1 = Counter(ngrams1)
            counter2 = Counter(ngrams2)
            sim = cosine_similarity(counter1, counter2)
            similarities.append(sim)

    return sum(similarities) / len(similarities) if similarities else 0.0


# ============== WAV TO MIDI CONVERSION ==============

def convert_wav_to_midi(wav_path: str, output_midi: str, api_url: str) -> bool:
    """Конвертує WAV в MIDI через API"""

    if os.path.exists(output_midi):
        print(f"  [CACHE] {output_midi}")
        return True

    print(f"  Конвертую: {wav_path}")

    try:
        with open(wav_path, "rb") as f:
            response = requests.post(
                f"{api_url}/process",
                files={"file": (os.path.basename(wav_path), f)},
                data={
                    "sound_type": "melody",
                    "extract_stem": "false",
                    "convert_midi": "true",
                },
                timeout=600
            )

        result = response.json()

        if "detail" in result:
            print(f"  Помилка API: {result['detail'][:100]}...")
            return False

        # Завантажуємо MIDI
        for url in result.get("download_urls", []):
            if url.endswith(".mid"):
                r = requests.get(f"{api_url}{url}")
                os.makedirs(os.path.dirname(output_midi), exist_ok=True)
                with open(output_midi, "wb") as f:
                    f.write(r.content)
                print(f"  Збережено: {output_midi}")
                return True

        print("  MIDI файл не знайдено у відповіді")
        return False

    except Exception as e:
        print(f"  Помилка: {e}")
        return False


def find_wav_file(directory: str, title: str) -> str:
    """Шукає WAV файл за назвою (часткове співпадіння)"""
    dir_path = Path(directory)

    if not dir_path.exists():
        return None

    # Очищаємо назву для пошуку
    clean_title = title.lower().replace("_", " ").replace("-", " ")

    for f in dir_path.iterdir():
        if f.suffix.lower() in ['.wav', '.mp3', '.flac']:
            clean_name = f.stem.lower().replace("_", " ").replace("-", " ")
            # Перевіряємо чи назва файлу містить частину title або навпаки
            if clean_title[:20] in clean_name or clean_name[:20] in clean_title:
                return str(f)

    # Якщо не знайшли, шукаємо перші слова
    title_words = clean_title.split()[:3]
    for f in dir_path.iterdir():
        if f.suffix.lower() in ['.wav', '.mp3', '.flac']:
            clean_name = f.stem.lower()
            if all(word in clean_name for word in title_words if len(word) > 2):
                return str(f)

    return None


# ============== MAIN PIPELINE ==============

def load_pairs(csv_path: str) -> List[Dict]:
    """Завантажує пари з CSV"""
    pairs = []
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            pairs.append({
                'id': row.get('', row.get('id', '')),
                'original': row['ori_title'],
                'comparison': row['comp_title'],
                'relation': row['relation'],
            })
    return pairs


def run_pipeline(api_url: str, original_dir: str, comparison_dir: str,
                 midi_dir: str, pairs_csv: str):
    """Основний пайплайн"""

    print("=" * 70)
    print("MIDI SIMILARITY PIPELINE")
    print("=" * 70)

    # Завантажуємо пари
    pairs = load_pairs(pairs_csv)
    print(f"\nЗавантажено {len(pairs)} пар з {pairs_csv}")

    # Створюємо папки для MIDI
    os.makedirs(f"{midi_dir}/original", exist_ok=True)
    os.makedirs(f"{midi_dir}/comparison", exist_ok=True)

    # Конвертуємо всі файли в MIDI
    print("\n" + "=" * 70)
    print("КРОК 1: Конвертація WAV -> MIDI")
    print("=" * 70)

    original_midis = {}  # title -> midi_path
    comparison_midis = {}

    # Збираємо всі оригінальні файли
    print("\n[ORIGINAL FILES]")
    for pair in pairs:
        title = pair['original']
        if title in original_midis:
            continue

        wav_path = find_wav_file(original_dir, title)
        if wav_path:
            midi_path = f"{midi_dir}/original/{Path(wav_path).stem}.mid"
            if convert_wav_to_midi(wav_path, midi_path, api_url):
                original_midis[title] = midi_path
        else:
            print(f"  [NOT FOUND] {title[:50]}...")

    # Збираємо всі comparison файли
    print("\n[COMPARISON FILES]")
    for pair in pairs:
        title = pair['comparison']
        if title in comparison_midis:
            continue

        wav_path = find_wav_file(comparison_dir, title)
        if wav_path:
            midi_path = f"{midi_dir}/comparison/{Path(wav_path).stem}.mid"
            if convert_wav_to_midi(wav_path, midi_path, api_url):
                comparison_midis[title] = midi_path
        else:
            print(f"  [NOT FOUND] {title[:50]}...")

    print(f"\nКонвертовано: {len(original_midis)} original, {len(comparison_midis)} comparison")

    # Порівнюємо кожен comparison з усіма original
    print("\n" + "=" * 70)
    print("КРОК 2: Пошук подібних пісень")
    print("=" * 70)

    results = []
    correct = 0
    total = 0

    for pair in pairs:
        comp_title = pair['comparison']
        expected_ori = pair['original']

        if comp_title not in comparison_midis:
            continue
        if expected_ori not in original_midis:
            continue

        total += 1
        comp_midi = comparison_midis[comp_title]

        print(f"\n[{total}] Шукаю подібність для: {comp_title[:50]}...")
        print(f"    Очікуваний оригінал: {expected_ori[:50]}...")

        # Рахуємо подібність з усіма оригіналами
        similarities = []
        for ori_title, ori_midi in original_midis.items():
            sim = calculate_similarity(comp_midi, ori_midi)
            similarities.append((ori_title, sim))

        # Сортуємо за подібністю
        similarities.sort(key=lambda x: x[1], reverse=True)

        # Топ-3 результати
        print(f"    Топ-3 подібні:")
        for i, (title, sim) in enumerate(similarities[:3]):
            marker = "✓" if title == expected_ori else " "
            print(f"      {i+1}. [{marker}] {sim:.4f} - {title[:40]}...")

        # Перевіряємо чи правильно знайдено
        best_match = similarities[0][0] if similarities else None
        is_correct = best_match == expected_ori

        if is_correct:
            correct += 1
            print(f"    ✅ ПРАВИЛЬНО!")
        else:
            print(f"    ❌ Неправильно (знайдено: {best_match[:40] if best_match else 'N/A'}...)")

        results.append({
            'comparison': comp_title,
            'expected': expected_ori,
            'found': best_match,
            'correct': is_correct,
            'similarity': similarities[0][1] if similarities else 0,
            'top3': similarities[:3],
        })

    # Підсумок
    print("\n" + "=" * 70)
    print("РЕЗУЛЬТАТИ")
    print("=" * 70)

    accuracy = correct / total if total > 0 else 0
    print(f"\nТочність (Top-1): {correct}/{total} = {accuracy:.2%}")

    # Top-3 accuracy
    top3_correct = sum(1 for r in results if r['expected'] in [t[0] for t in r['top3']])
    top3_accuracy = top3_correct / total if total > 0 else 0
    print(f"Точність (Top-3): {top3_correct}/{total} = {top3_accuracy:.2%}")

    # Деталі
    print("\n" + "-" * 70)
    print("Детальні результати:")
    print("-" * 70)

    for r in results:
        status = "✅" if r['correct'] else "❌"
        print(f"{status} {r['comparison'][:35]:35} -> {r['found'][:30] if r['found'] else 'N/A':30} ({r['similarity']:.3f})")

    return results, accuracy


# ============== RUN ==============

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description='MIDI Similarity Pipeline')
    parser.add_argument('--api', default=API_URL, help='API URL')
    parser.add_argument('--original', default=ORIGINAL_DIR, help='Original files directory')
    parser.add_argument('--comparison', default=COMPARISON_DIR, help='Comparison files directory')
    parser.add_argument('--midi', default=MIDI_DIR, help='MIDI cache directory')
    parser.add_argument('--pairs', default=PAIRS_CSV, help='Pairs CSV file')

    args = parser.parse_args()

    results, accuracy = run_pipeline(
        api_url=args.api,
        original_dir=args.original,
        comparison_dir=args.comparison,
        midi_dir=args.midi,
        pairs_csv=args.pairs,
    )